In [1]:
import os
import sys

python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

from pyspark.sql import SparkSession
from pyspark.sql.functions import count, avg, sum, max, min

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Test")
    .config("spark.pyspark.python", python_path)
    .config("spark.pyspark.driver.python", python_path)
    .getOrCreate()
)

In [2]:
DATA_DIR = r"C:\Users\Atul kumar\Documents\pyspark_ecommerce_5k\data"

orders = spark.read.csv(
    os.path.join(DATA_DIR, "orders.csv"),
    header=True,
    inferSchema=True
)

customers = spark.read.csv(
    os.path.join(DATA_DIR, "customers.csv"),
    header=True,
    inferSchema=True
)

products = spark.read.csv(
    os.path.join(DATA_DIR, "products.csv"),
    header=True,
    inferSchema=True
)

print("Orders:", orders.count())
print("Customers:", customers.count())
print("Products:", products.count())

Orders: 5000
Customers: 1000
Products: 20


In [3]:
print("Orders:", orders.count())
print("Customers:", customers.count())
print("Products:", products.count())

orders.show(5, truncate=False)
customers.show(5, truncate=False)
products.show(5, truncate=False)

Orders: 5000
Customers: 1000
Products: 20
+--------+-----------+----------+----------+--------+---------+----------------+
|order_id|customer_id|product_id|order_date|quantity|status   |payment_method  |
+--------+-----------+----------+----------+--------+---------+----------------+
|1       |481        |118       |2025-03-26|3       |Delivered|Cash on Delivery|
|2       |243        |112       |2026-06-29|4       |Delivered|Net Banking     |
|3       |636        |115       |2026-07-28|4       |Shipped  |Cash on Delivery|
|4       |81         |120       |2025-12-29|2       |Delivered|Debit Card      |
|5       |473        |114       |2026-07-14|1       |Shipped  |UPI             |
+--------+-----------+----------+----------+--------+---------+----------------+
only showing top 5 rows
+-----------+-------------+---------+--------------+------+---------------------+
|customer_id|customer_name|city     |state         |gender|email                |
+-----------+-------------+---------+----

In [4]:
orders.printSchema()
customers.printSchema()
products.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- email: string (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)



In [5]:
orders.printSchema()
customers.printSchema()
products.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- email: string (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)



In [6]:
from pyspark.sql.functions import col, count, sum as spark_sum

print("Duplicate order IDs:")
print(orders.count() - orders.select("order_id").distinct().count())

print("Invalid quantities:")
orders.filter(col("quantity") <= 0).show()

print("Invalid customer IDs:")
orders.join(
    customers.select("customer_id"),
    on="customer_id",
    how="left_anti"
).show()

Duplicate order IDs:
0
Invalid quantities:
+--------+-----------+----------+----------+--------+---------+--------------+
|order_id|customer_id|product_id|order_date|quantity|   status|payment_method|
+--------+-----------+----------+----------+--------+---------+--------------+
|     102|        917|       115|2026-09-17|       0| Returned|   Credit Card|
|     556|        571|       114|2026-09-23|       0|Delivered|   Net Banking|
|    1000|        319|       119|2026-09-07|       0|  Shipped|   Credit Card|
+--------+-----------+----------+----------+--------+---------+--------------+

Invalid customer IDs:
+-----------+--------+----------+----------+--------+---------+--------------+
|customer_id|order_id|product_id|order_date|quantity|   status|payment_method|
+-----------+--------+----------+----------+--------+---------+--------------+
|      99999|     223|       111|2026-04-21|       1|Delivered|   Credit Card|
|      99999|     778|       109|2025-05-04|       2|Delivered|  

In [7]:
from pyspark.sql.functions import to_date

clean_orders = (
    orders
    .dropDuplicates(["order_id"])
    .filter(col("quantity") > 0)
    .join(
        customers.select("customer_id").distinct(),
        on="customer_id",
        how="inner"
    )
    .join(
        products.select("product_id").distinct(),
        on="product_id",
        how="inner"
    )
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
)

print("Raw orders:", orders.count())
print("Clean orders:", clean_orders.count())

clean_orders.show(10)

Raw orders: 5000
Clean orders: 4995
+----------+-----------+--------+----------+--------+---------+----------------+
|product_id|customer_id|order_id|order_date|quantity|   status|  payment_method|
+----------+-----------+--------+----------+--------+---------+----------------+
|       118|        481|       1|2025-03-26|       3|Delivered|Cash on Delivery|
|       112|        243|       2|2026-06-29|       4|Delivered|     Net Banking|
|       115|        636|       3|2026-07-28|       4|  Shipped|Cash on Delivery|
|       120|         81|       4|2025-12-29|       2|Delivered|      Debit Card|
|       114|        473|       5|2026-07-14|       1|  Shipped|             UPI|
|       108|        867|       6|2025-12-11|       4|Delivered|      Debit Card|
|       106|         96|       7|2025-10-16|       1|  Shipped|             UPI|
|       117|        792|       8|2025-07-09|       3|  Shipped|     Net Banking|
|       101|        206|       9|2025-10-12|       2|Delivered|      Debi

In [8]:
sales = (
    clean_orders
    .join(customers, on="customer_id", how="inner")
    .join(products, on="product_id", how="inner")
    .withColumn(
        "revenue",
        col("quantity") * col("price")
    )
)

sales.select(
    "order_id",
    "order_date",
    "customer_name",
    "city",
    "product_name",
    "category",
    "quantity",
    "price",
    "revenue",
    "status"
).show(10, truncate=False)

+--------+----------+-------------+---------+--------------------+-----------+--------+-------+-------+---------+
|order_id|order_date|customer_name|city     |product_name        |category   |quantity|price  |revenue|status   |
+--------+----------+-------------+---------+--------------------+-----------+--------+-------+-------+---------+
|1       |2025-03-26|Isha Pawar   |Nagpur   |Lunch Box           |Kitchen    |3       |483.89 |1451.67|Delivered|
|2       |2026-06-29|Arjun Sharma |Hyderabad|Power Bank          |Electronics|4       |1676.39|6705.56|Delivered|
|3       |2026-07-28|Kavya Nair   |Bengaluru|Wall Clock          |Home       |4       |1415.34|5661.36|Shipped  |
|4       |2025-12-29|Priya Mishra |Lucknow  |Travel Pouch        |Accessories|2       |862.54 |1725.08|Delivered|
|5       |2026-07-14|Sara Reddy   |Nagpur   |Yoga Mat            |Fitness    |1       |537.21 |537.21 |Shipped  |
|6       |2025-12-11|Vikram Gupta |Chennai  |Backpack            |Accessories|4       |7

In [9]:
sales.createOrReplaceTempView("sales")

In [10]:
spark.sql("""
SELECT ROUND(SUM(revenue), 2) AS total_revenue
FROM sales
WHERE status = 'Delivered'
""").show()

+-------------+
|total_revenue|
+-------------+
|   5963439.07|
+-------------+



In [11]:
spark.sql("""
SELECT
    category,
    ROUND(SUM(revenue), 2) AS total_revenue
FROM sales
WHERE status = 'Delivered'
GROUP BY category
ORDER BY total_revenue DESC
""").show()

+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|   Footwear|   1413750.77|
|Electronics|   1280932.93|
|       Home|     957392.1|
|   Clothing|    808643.76|
|    Kitchen|    676079.58|
|Accessories|    629483.86|
|    Fitness|    197156.07|
+-----------+-------------+



In [12]:
spark.sql("""
SELECT
    customer_name,
    COUNT(*) AS total_orders,
    ROUND(SUM(revenue), 2) AS total_revenue
FROM sales
WHERE status = 'Delivered'
GROUP BY customer_id, customer_name
ORDER BY total_revenue DESC
LIMIT 10
""").show()

+-------------+------------+-------------+
|customer_name|total_orders|total_revenue|
+-------------+------------+-------------+
|   Kavya Nair|           6|     39456.46|
|    Dev Kumar|           7|     33248.68|
|  Karan Joshi|           8|     28538.55|
| Nikhil Verma|           8|     27558.69|
|  Priya Gupta|           4|     26307.68|
|   Atul Singh|           6|     24934.77|
|     Amit Das|           4|     24918.42|
|    Kavya Das|           8|     24893.08|
|  Riya Mishra|           8|     24098.31|
|   Dev Sharma|           5|     23074.33|
+-------------+------------+-------------+



In [13]:
spark.sql("""
SELECT
    payment_method,
    COUNT(*) AS total_orders,
    ROUND(SUM(revenue), 2) AS total_revenue
FROM sales
GROUP BY payment_method
ORDER BY total_revenue DESC
""").show()

+----------------+------------+-------------+
|  payment_method|total_orders|total_revenue|
+----------------+------------+-------------+
|      Debit Card|        1033|   1874248.55|
|     Credit Card|        1043|   1739000.95|
|             UPI|         966|    1721776.6|
|Cash on Delivery|         978|   1706708.37|
|     Net Banking|         975|    1596565.4|
+----------------+------------+-------------+



In [14]:
spark.sql("""
SELECT
    order_id,
    revenue,
    CASE
        WHEN revenue >= 3000 THEN 'High Value'
        WHEN revenue >= 1000 THEN 'Medium Value'
        ELSE 'Low Value'
    END AS order_category
FROM sales
""").show(10)

+--------+-------+--------------+
|order_id|revenue|order_category|
+--------+-------+--------------+
|     148| 862.54|     Low Value|
|     463|2734.44|  Medium Value|
|     471| 3254.8|    High Value|
|     496|1253.76|  Medium Value|
|     833| 862.54|     Low Value|
|    1088| 415.32|     Low Value|
|    1238|2830.68|  Medium Value|
|    1342| 862.54|     Low Value|
|    1580|1888.08|  Medium Value|
|    1591|3853.42|    High Value|
+--------+-------+--------------+
only showing top 10 rows


In [15]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank

window_spec = Window.partitionBy("customer_id").orderBy(
    col("revenue").desc()
)

ranked_sales = (
    sales
    .withColumn("row_num", row_number().over(window_spec))
    .withColumn("rank_num", rank().over(window_spec))
    .withColumn("dense_rank_num", dense_rank().over(window_spec))
)

ranked_sales.select(
    "customer_id",
    "customer_name",
    "order_id",
    "revenue",
    "row_num",
    "rank_num",
    "dense_rank_num"
).orderBy("customer_id", "row_num").show(20)

+-----------+-------------+--------+-------+-------+--------+--------------+
|customer_id|customer_name|order_id|revenue|row_num|rank_num|dense_rank_num|
+-----------+-------------+--------+-------+-------+--------+--------------+
|          1| Nikhil Verma|    1213|8381.95|      1|       1|             1|
|          1| Nikhil Verma|     943|7706.84|      2|       2|             2|
|          1| Nikhil Verma|    3423|3776.16|      3|       3|             3|
|          1| Nikhil Verma|    1866|2805.76|      4|       4|             4|
|          1| Nikhil Verma|     360|2186.16|      5|       5|             5|
|          1| Nikhil Verma|    4072|1074.42|      6|       6|             6|
|          1| Nikhil Verma|    3716| 989.27|      7|       7|             7|
|          1| Nikhil Verma|    2218|  813.7|      8|       8|             8|
|          1| Nikhil Verma|    1421|  813.7|      9|       8|             8|
|          1| Nikhil Verma|    1170| 313.44|     10|      10|             9|

In [16]:
top_customer_orders = ranked_sales.filter(col("row_num") == 1)

top_customer_orders.select(
    "customer_name", "order_id", "revenue"
).show(20)

+-------------+--------+-------+
|customer_name|order_id|revenue|
+-------------+--------+-------+
| Nikhil Verma|    1213|8381.95|
| Ananya Mehta|    4014|2967.81|
|    Dev Pawar|     575|7706.84|
|  Kavya Patel|    2196|7706.84|
|      Dev Das|    1546|6705.56|
|   Arjun Shah|    4412|1402.88|
| Simran Kumar|     224|1725.08|
| Nikhil Pawar|    1190|2587.62|
|   Isha Yadav|     401| 2076.6|
|  Rahul Pawar|    3212|7706.84|
|   Amit Pawar|    1690|7706.84|
|    Dev Verma|    4808| 1750.4|
|  Priya Yadav|    1489| 1627.4|
|Nikhil Sharma|    3473|5029.17|
| Ananya Yadav|    1091|2805.76|
|  Rahul Kumar|    2685|2186.16|
| Simran Joshi|    2549|1611.63|
|    Neha Shah|    4105|1611.63|
| Simran Verma|    1766|3853.42|
| Meera Sharma|    2572|7706.84|
+-------------+--------+-------+
only showing top 20 rows


In [17]:
from pyspark.sql.functions import year, month, round as spark_round

monthly_revenue = (
    sales
    .filter(col("status") == "Delivered")
    .withColumn("year", year("order_date"))
    .withColumn("month", month("order_date"))
    .groupBy("year", "month")
    .agg(
        spark_round(spark_sum("revenue"), 2).alias("total_revenue"),
        count("*").alias("delivered_orders")
    )
    .orderBy("year", "month")
)

monthly_revenue.show(50)

+----+-----+-------------+----------------+
|year|month|total_revenue|delivered_orders|
+----+-----+-------------+----------------+
|2025|    1|    344239.97|             176|
|2025|    2|    271386.83|             133|
|2025|    3|    274868.42|             146|
|2025|    4|     274637.1|             181|
|2025|    5|    286657.56|             162|
|2025|    6|    191383.62|             140|
|2025|    7|    257505.53|             152|
|2025|    8|    218614.73|             137|
|2025|    9|    265005.81|             164|
|2025|   10|    258800.91|             150|
|2025|   11|    283160.61|             187|
|2025|   12|    242409.95|             160|
|2026|    1|    261779.54|             169|
|2026|    2|     268610.0|             150|
|2026|    3|    299882.81|             171|
|2026|    4|    344951.72|             183|
|2026|    5|    252956.76|             154|
|2026|    6|    261125.51|             136|
|2026|    7|    290369.16|             168|
|2026|    8|    284711.73|      

In [18]:
pivot_df = (
    sales
    .groupBy("category")
    .pivot("status", ["Delivered", "Shipped", "Cancelled", "Returned"])
    .agg(spark_sum("revenue"))
)

pivot_df.show(truncate=False)

+-----------+------------------+------------------+------------------+------------------+
|category   |Delivered         |Shipped           |Cancelled         |Returned          |
+-----------+------------------+------------------+------------------+------------------+
|Kitchen    |676079.5800000007 |166916.1900000001 |97206.99999999999 |48621.64000000001 |
|Home       |957392.1          |208456.08000000005|128882.15999999997|72027.36          |
|Electronics|1280932.929999999 |270166.0400000002 |222074.2900000001 |65675.59999999999 |
|Clothing   |808643.7600000018 |202697.2599999999 |94687.31999999998 |73927.68999999999 |
|Fitness    |197156.07000000007|43514.00999999998 |17727.93          |11818.619999999999|
|Footwear   |1413750.7699999996|315167.0099999997 |235241.94999999998|110443.06999999998|
|Accessories|629483.859999997  |149884.6200000001 |95563.52000000006 |44161.44          |
+-----------+------------------+------------------+------------------+------------------+



In [19]:
print("Current partitions:", sales.rdd.getNumPartitions())

Current partitions: 1


In [20]:
sales_repartitioned = sales.repartition(4, "category")

print("New partitions:", sales_repartitioned.rdd.getNumPartitions())

New partitions: 4


In [21]:
sales.cache()

print("Cached dataset rows:", sales.count())

sales.unpersist()

Cached dataset rows: 4995


DataFrame[product_id: int, customer_id: int, order_id: int, order_date: date, quantity: int, status: string, payment_method: string, customer_name: string, city: string, state: string, gender: string, email: string, product_name: string, category: string, price: double, revenue: double]

In [22]:
sales.filter(col("status") == "Delivered").explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (34)
+- Project (33)
   +- BroadcastHashJoin Inner BuildRight (32)
      :- Project (28)
      :  +- BroadcastHashJoin Inner BuildRight (27)
      :     :- Project (23)
      :     :  +- BroadcastHashJoin Inner BuildRight (22)
      :     :     :- Project (15)
      :     :     :  +- BroadcastHashJoin Inner BuildRight (14)
      :     :     :     :- Filter (7)
      :     :     :     :  +- SortAggregate (6)
      :     :     :     :     +- Sort (5)
      :     :     :     :        +- Exchange (4)
      :     :     :     :           +- SortAggregate (3)
      :     :     :     :              +- Sort (2)
      :     :     :     :                 +- Scan csv  (1)
      :     :     :     +- BroadcastExchange (13)
      :     :     :        +- HashAggregate (12)
      :     :     :           +- Exchange (11)
      :     :     :              +- HashAggregate (10)
      :     :     :                 +- Filter (9)
      :     :     :                    +- 

In [24]:
print("========== PROJECT SUMMARY ==========")

print("Raw orders:", orders.count())
print("Clean orders:", clean_orders.count())
print("Final sales rows:", sales.count())

print("Unique customers in sales:",
      sales.select("customer_id").distinct().count())

print("Unique products sold:",
      sales.select("product_id").distinct().count())

print("Total revenue across all statuses:",
      sales.agg(spark_round(spark_sum("revenue"), 2)).first()[0])

print("Delivered revenue:",
      sales.filter(col("status") == "Delivered")
           .agg(spark_round(spark_sum("revenue"), 2))
           .first()[0])

print("======================================")

========== PROJECT SUMMARY ==========
Raw orders: 5000
Clean orders: 4995
Final sales rows: 4995
Unique customers in sales: 998
Unique products sold: 20
Total revenue across all statuses: 8638299.87
Delivered revenue: 5963439.07


In [25]:
import os

# Output folder ka path
OUTPUT_DIR = os.path.abspath("output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Complete processed data - Parquet
sales.write \
    .mode("overwrite") \
    .parquet(os.path.join(OUTPUT_DIR, "sales_parquet"))

print("Processed Parquet data saved!")

# 2. Complete processed data - CSV
sales.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(os.path.join(OUTPUT_DIR, "sales_csv"))

print("Processed CSV data saved!")

Py4JJavaError: An error occurred while calling o316.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdir

In [26]:
import os

# Output folder
OUTPUT_DIR = os.path.abspath("output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# PySpark DataFrame ko Pandas DataFrame mein convert karke CSV save karo
sales.toPandas().to_csv(
    os.path.join(OUTPUT_DIR, "sales_final.csv"),
    index=False
)

print("CSV file successfully saved!")
print("File location:", os.path.join(OUTPUT_DIR, "sales_final.csv"))

CSV file successfully saved!
File location: C:\Users\Atul kumar\output\sales_final.csv


In [27]:
import os

print("Notebook folder:", os.getcwd())
print("Notebook file:", os.path.abspath("PySparkSalesProject.ipynb"))

Notebook folder: C:\Users\Atul kumar
Notebook file: C:\Users\Atul kumar\PySparkSalesProject.ipynb
